In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

2025-03-11 15:39:20.719982: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-11 15:39:34.222242: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-11 15:39:34.222304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-11 15:39:37.738552: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-11 15:39:40.627732: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the models required
frozen_model = tf.keras.models.load_model('models/VGG19_CNN_LSTM_frozen.keras')
finetuned_model = tf.keras.models.load_model('models/VGG19_CNN_LSTM_finetuned.keras')

2025-03-11 15:42:10.607655: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_count']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

Block 0103

Predictions wth the frozen model

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-11 16:28:38.675493: W tensorflow/core/kernels/gpu_utils.cc:54] Failed to allocate memory for convolution redzone checking; skipping this check. This is benign and only means that we won't check cudnn for out-of-bounds reads and writes. This message will only be printed once.
2025-03-11 16:28:39.942325: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 295s 526ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'frozen_block_0103_VGG19')

In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[0.0, 74.72842990390428, 95.79638231754828, 0.0, 0.0, 0.35355188942594395, 0.0]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

[37.16732290457524,
 38.107001730603564,
 PearsonRResult(statistic=0.5347620679533426, pvalue=0.21619175949498035),
 -53.234020932861995]

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,0.000000
1,Block0103_2020_08_27,39,39.000001,74.728430
2,Block0103_2020_08_28,41,41.000000,95.796382
3,Block0103_2020_08_31,31,31.000000,0.000000
4,Block0103_2020_09_02,32,32.000000,0.000000
5,Block0103_2020_09_07,40,40.002086,0.353552
6,Block0103_2020_09_16,27,27.000176,0.000000


Predictions with the finetuned model

In [14]:
%%time
# first get the predictions
finetuned_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', finetuned_model)

384/384 [==============================] - 203s 527ms/step
CPU times: user 3min 42s, sys: 6.96 s, total: 3min 49s
Wall time: 5min 36s


In [15]:
finetuned_preds_block_0103.shape

(12288, 7)

In [16]:
finetuned_final_forecasts_block_0103 = get_final_forecasted_and_true_values(finetuned_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'finetuned_block_0103_VGG19')

In [17]:
finetuned_normalized_forecasts_block_0103 = finetuned_final_forecasts_block_0103[0]

In [18]:
print(finetuned_normalized_forecasts_block_0103)

[0.0, 76.56449073500532, 80.54768772104335, 64.31793303584415, 0.0, 61.95321012664877, 30.49875956770019]


In [19]:
mae_finetuned_block_0103 = finetuned_final_forecasts_block_0103[1]
mae_finetuned_block_0103

[29.697440169463114,
 32.07887596761597,
 PearsonRResult(statistic=0.32225060869116506, pvalue=0.48088761455534246),
 -37.432667594465535]

In [20]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0103[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,0.000000
1,Block0103_2020_08_27,39,39.000001,76.564491
2,Block0103_2020_08_28,41,41.000000,80.547688
3,Block0103_2020_08_31,31,31.000000,64.317933
4,Block0103_2020_09_02,32,32.000000,0.000000
5,Block0103_2020_09_07,40,40.002086,61.953210
6,Block0103_2020_09_16,27,27.000176,30.498760


Block 0104

Predictions wth the frozen model

In [21]:
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 202s 528ms/step


In [22]:
frozen_preds_block_0104.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'frozen_block_0104_VGG19')

In [24]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [25]:
print(frozen_normalized_forecasts_block_0104)

[0.0, 55.870897802703794, 70.93123323647899, 0.0, 0.0, 1.1670305265544505, 0.0]


In [26]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

[34.66215721608976,
 35.104412431527955,
 PearsonRResult(statistic=-0.17346133482520354, pvalue=0.7099330611763571),
 -51.05488692756717]

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,0.000000
1,Block0104_2020_08_27,30,30.000000,55.870898
2,Block0104_2020_08_28,39,39.000001,70.931233
3,Block0104_2020_08_31,40,40.000000,0.000000
4,Block0104_2020_09_02,41,40.998810,0.000000
5,Block0104_2020_09_07,42,42.169009,1.167031
6,Block0104_2020_09_16,30,30.005317,0.000000


Predictions with the finetuned model

In [28]:
# first get the predictions
finetuned_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', finetuned_model)

384/384 [==============================] - 202s 526ms/step


In [29]:
finetuned_preds_block_0104.shape

(12288, 7)

In [30]:
finetuned_final_forecasts_block_0104 = get_final_forecasted_and_true_values(finetuned_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'finetuned_block_0104_VGG19')

In [31]:
finetuned_normalized_forecasts_block_0104 = finetuned_final_forecasts_block_0104[0]

In [32]:
print(finetuned_normalized_forecasts_block_0104)

[0.0, 39.99760877372956, 45.71411256163103, 39.23417068301554, 0.0, 36.173953786670616, 22.67953848931109]


In [33]:
mae_finetuned_block_0104 = finetuned_final_forecasts_block_0104[1]
mae_finetuned_block_0104

[14.946294053766195,
 20.712832304205204,
 PearsonRResult(statistic=0.09268269227533679, pvalue=0.843331897815178),
 -17.122456621589834]

In [34]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0104[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,0.000000
1,Block0104_2020_08_27,30,30.000000,39.997609
2,Block0104_2020_08_28,39,39.000001,45.714113
3,Block0104_2020_08_31,40,40.000000,39.234171
4,Block0104_2020_09_02,41,40.998810,0.000000
5,Block0104_2020_09_07,42,42.169009,36.173954
6,Block0104_2020_09_16,30,30.005317,22.679538


Block 0105

Predictions wth the frozen model

In [35]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [36]:
frozen_preds_block_0105.shape

(12288, 7)

In [37]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'frozen_block_0105_VGG19')

In [38]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [39]:
print(frozen_normalized_forecasts_block_0105)

[0.0, 72.5758405856678, 92.55068297138513, 0.0, 0.0, 0.5334645590652217, 0.0]


In [40]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

[34.51329414256968,
 35.15049305243928,
 PearsonRResult(statistic=0.7724501552860225, pvalue=0.04178703054752738),
 -12.036671173481807]

In [41]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,0.000000
1,Block0105_2020_08_27,46,46.002743,72.575841
2,Block0105_2020_08_28,58,58.000696,92.550683
3,Block0105_2020_08_31,41,41.000032,0.000000
4,Block0105_2020_09_02,41,41.001190,0.000000
5,Block0105_2020_09_07,36,36.000022,0.533465
6,Block0105_2020_09_16,23,23.000000,0.000000


Predictions with the finetuned model

In [42]:
# first get the predictions
finetuned_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [43]:
finetuned_preds_block_0105.shape

(12288, 7)

In [44]:
finetuned_final_forecasts_block_0105 = get_final_forecasted_and_true_values(finetuned_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'finetuned_block_0105_VGG19')

In [45]:
finetuned_normalized_forecasts_block_0105 = finetuned_final_forecasts_block_0105[0]

In [46]:
print(finetuned_normalized_forecasts_block_0105)

[0.0, 65.8343384796018, 74.12152195938222, 58.2237832353138, 0.0, 53.97242144180493, 27.023262769122663]


In [47]:
mae_finetuned_block_0105 = finetuned_final_forecasts_block_0105[1]
mae_finetuned_block_0105

[22.310761126460772,
 25.551510265508185,
 PearsonRResult(statistic=0.46823632909791757, pvalue=0.28929084682203693),
 -5.888695987418203]

In [48]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0105[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,0.000000
1,Block0105_2020_08_27,46,46.002743,65.834338
2,Block0105_2020_08_28,58,58.000696,74.121522
3,Block0105_2020_08_31,41,41.000032,58.223783
4,Block0105_2020_09_02,41,41.001190,0.000000
5,Block0105_2020_09_07,36,36.000022,53.972421
6,Block0105_2020_09_16,23,23.000000,27.023263


Block 0106

Predictions wth the frozen model

In [49]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [50]:
frozen_preds_block_0106.shape

(12288, 7)

In [51]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'frozen_block_0106_VGG19')

In [52]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [53]:
print(frozen_normalized_forecasts_block_0106)

[0.0, 61.250757920197024, 78.19853725991098, 8.095080556813627e-05, 0.0, 1.0368581162348771, 0.0]


In [54]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

[37.48747944472394,
 38.203906538762816,
 PearsonRResult(statistic=0.12676006851833135, pvalue=0.7865301346871597),
 -121.46127614093096]

In [55]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,0.000000
1,Block0106_2020_08_27,39,38.999989,61.250758
2,Block0106_2020_08_28,45,45.000000,78.198537
3,Block0106_2020_08_31,40,39.986887,0.000081
4,Block0106_2020_09_02,43,42.999997,0.000000
5,Block0106_2020_09_07,48,47.996502,1.036858
6,Block0106_2020_09_16,38,38.000000,0.000000


Predictions with the finetuned model

In [56]:
# first get the predictions
finetuned_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [57]:
finetuned_preds_block_0106.shape

(12288, 7)

In [58]:
finetuned_final_forecasts_block_0106 = get_final_forecasted_and_true_values(finetuned_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'finetuned_block_0106_VGG19')

In [59]:
finetuned_normalized_forecasts_block_0106 = finetuned_final_forecasts_block_0106[0]

In [60]:
print(finetuned_normalized_forecasts_block_0106)

[0.0, 59.81350674031757, 60.51195927642835, 49.94408034867084, 0.0, 48.71722017619899, 26.786957278075462]


In [61]:
mae_finetuned_block_0106 = finetuned_final_forecasts_block_0106[1]
mae_finetuned_block_0106

[20.028544180505758,
 24.695153975942297,
 PearsonRResult(statistic=0.2663581451914392, pvalue=0.5636846531087694),
 -50.168974083697655]

In [62]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0106[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,0.000000
1,Block0106_2020_08_27,39,38.999989,59.813507
2,Block0106_2020_08_28,45,45.000000,60.511959
3,Block0106_2020_08_31,40,39.986887,49.944080
4,Block0106_2020_09_02,43,42.999997,0.000000
5,Block0106_2020_09_07,48,47.996502,48.717220
6,Block0106_2020_09_16,38,38.000000,26.786957


Block 0201

Predictions wth the frozen model

In [63]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

384/384 [==============================] - 202s 528ms/step


In [64]:
frozen_preds_block_0201.shape

(12288, 7)

In [65]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'frozen_block_0201_VGG19')

In [66]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [67]:
print(frozen_normalized_forecasts_block_0201)

[0.0, 67.24212251159567, 86.17233449466944, 0.0, 0.0, 0.7486796368787559, 0.0]


In [68]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

[35.66653962419805,
 36.40317244848755,
 PearsonRResult(statistic=0.6212901637742956, pvalue=0.13642866504682824),
 -35.68607754316489]

In [69]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,0.000000
1,Block0201_2020_08_27,45,45.000040,67.242123
2,Block0201_2020_08_28,47,47.000001,86.172334
3,Block0201_2020_08_31,38,38.000000,0.000000
4,Block0201_2020_09_02,42,42.000041,0.000000
5,Block0201_2020_09_07,35,35.000000,0.748680
6,Block0201_2020_09_16,29,29.000000,0.000000


Predictions with the finetuned model

In [70]:
# first get the predictions
finetuned_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [71]:
finetuned_preds_block_0201.shape

(12288, 7)

In [72]:
finetuned_final_forecasts_block_0201 = get_final_forecasted_and_true_values(finetuned_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'finetuned_block_0201_VGG19')

In [73]:
finetuned_normalized_forecasts_block_0201 = finetuned_final_forecasts_block_0201[0]

In [74]:
print(finetuned_normalized_forecasts_block_0201)

[0.0, 62.33313782797693, 67.38100317196059, 54.96745772662171, 0.0, 51.4626142377649, 28.008748437747386]


In [75]:
mae_finetuned_block_0201 = finetuned_final_forecasts_block_0201[1]
mae_finetuned_block_0201

[22.733637789510965,
 26.89853375229888,
 PearsonRResult(statistic=0.06303794994254634, pvalue=0.893195955892664),
 -19.02995750460709]

In [76]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0201[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,0.000000
1,Block0201_2020_08_27,45,45.000040,62.333138
2,Block0201_2020_08_28,47,47.000001,67.381003
3,Block0201_2020_08_31,38,38.000000,54.967458
4,Block0201_2020_09_02,42,42.000041,0.000000
5,Block0201_2020_09_07,35,35.000000,51.462614
6,Block0201_2020_09_16,29,29.000000,28.008748


Block 0202

Predictions wth the frozen model

In [77]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [78]:
frozen_preds_block_0202.shape

(12288, 7)

In [79]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'frozen_block_0202_VGG19')

In [80]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [81]:
print(frozen_normalized_forecasts_block_0202)

[0.0, 75.53737031206627, 97.86266491121347, 0.0, 0.0, 0.5212034890477703, 0.0]


In [82]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

[32.125547390604574,
 38.571365555357914,
 PearsonRResult(statistic=0.717619809723237, pvalue=0.06941749710991191),
 -432.92715356813983]

In [83]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,0.000000
1,Block0202_2020_08_27,21,21.000000,75.537370
2,Block0202_2020_08_28,23,23.000000,97.862665
3,Block0202_2020_08_31,21,20.999982,0.000000
4,Block0202_2020_09_02,21,21.000000,0.000000
5,Block0202_2020_09_07,18,18.000000,0.521203
6,Block0202_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [84]:
# first get the predictions
finetuned_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [85]:
finetuned_preds_block_0202.shape

(12288, 7)

In [86]:
finetuned_final_forecasts_block_0202 = get_final_forecasted_and_true_values(finetuned_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'finetuned_block_0202_VGG19')

In [87]:
finetuned_normalized_forecasts_block_0202 = finetuned_final_forecasts_block_0202[0]

In [88]:
print(finetuned_normalized_forecasts_block_0202)

[0.0, 81.91487312157804, 88.57823776321298, 67.47983383518145, 0.0, 65.07385857474877, 31.268922056618937]


In [89]:
mae_finetuned_block_0202 = finetuned_final_forecasts_block_0202[1]
mae_finetuned_block_0202

[38.90224647876288,
 43.63569057312177,
 PearsonRResult(statistic=0.4967448262973031, pvalue=0.2567710047285169),
 -554.3547684396916]

In [90]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0202[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,0.000000
1,Block0202_2020_08_27,21,21.000000,81.914873
2,Block0202_2020_08_28,23,23.000000,88.578238
3,Block0202_2020_08_31,21,20.999982,67.479834
4,Block0202_2020_09_02,21,21.000000,0.000000
5,Block0202_2020_09_07,18,18.000000,65.073859
6,Block0202_2020_09_16,18,18.000000,31.268922


Block 0205

Predictions wth the frozen model

In [91]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [92]:
frozen_preds_block_0205.shape

(12288, 7)

In [93]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'frozen_block_0205_VGG19')

In [94]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [95]:
print(frozen_normalized_forecasts_block_0205)

[0.0, 56.92528881387535, 72.52650989176138, 8.332028361716463e-05, 0.0, 1.4617265322536923, 0.0]


In [96]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

[34.284284121871345,
 35.604455773062355,
 PearsonRResult(statistic=0.4385819951005365, pvalue=0.32493332166374184),
 -76.25893815161903]

In [97]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,0.000000
1,Block0205_2020_08_27,42,42.000001,56.925289
2,Block0205_2020_08_28,45,45.000000,72.526510
3,Block0205_2020_08_31,43,43.000000,0.000083
4,Block0205_2020_09_02,39,39.000000,0.000000
5,Block0205_2020_09_07,41,40.999915,1.461727
6,Block0205_2020_09_16,32,31.999656,0.000000


Predictions with the finetuned model

In [98]:
# first get the predictions
finetuned_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [99]:
finetuned_preds_block_0205.shape

(12288, 7)

In [100]:
finetuned_final_forecasts_block_0205 = get_final_forecasted_and_true_values(finetuned_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'finetuned_block_0205_VGG19')

In [101]:
finetuned_normalized_forecasts_block_0205 = finetuned_final_forecasts_block_0205[0]

In [102]:
print(finetuned_normalized_forecasts_block_0205)

[0.0, 56.408415888665026, 55.204839541825216, 45.617720590096845, 0.0, 44.8334248939204, 25.277745464261184]


In [103]:
mae_finetuned_block_0205 = finetuned_final_forecasts_block_0205[1]
mae_finetuned_block_0205

[17.25523649289233,
 23.40780551148191,
 PearsonRResult(statistic=0.2706299607054956, pvalue=0.5572024922475312),
 -32.393460925752215]

In [104]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0205[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,0.000000
1,Block0205_2020_08_27,42,42.000001,56.408416
2,Block0205_2020_08_28,45,45.000000,55.204840
3,Block0205_2020_08_31,43,43.000000,45.617721
4,Block0205_2020_09_02,39,39.000000,0.000000
5,Block0205_2020_09_07,41,40.999915,44.833425
6,Block0205_2020_09_16,32,31.999656,25.277745


Block 0206

Predictions wth the frozen model

In [105]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [106]:
frozen_preds_block_0206.shape

(12288, 7)

In [107]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'frozen_block_0206_VGG19')

In [108]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [109]:
print(frozen_normalized_forecasts_block_0206)

[0.0, 71.98828139338002, 92.74915607998558, 0.0, 0.0, 0.5495285360805534, 0.0]


In [110]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

[31.741129848183583,
 33.692501219943914,
 PearsonRResult(statistic=0.6235565210703164, pvalue=0.13458394251718772),
 -13.350889392244632]

In [111]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,0.000000
1,Block0206_2020_08_27,42,41.998810,71.988281
2,Block0206_2020_08_28,39,39.000067,92.749156
3,Block0206_2020_08_31,32,32.000003,0.000000
4,Block0206_2020_09_02,25,25.000352,0.000000
5,Block0206_2020_09_07,23,23.000040,0.549529
6,Block0206_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [112]:
# first get the predictions
finetuned_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', finetuned_model)

384/384 [==============================] - 202s 528ms/step


In [113]:
finetuned_preds_block_0206.shape

(12288, 7)

In [114]:
finetuned_final_forecasts_block_0206 = get_final_forecasted_and_true_values(finetuned_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'finetuned_block_0206_VGG19')

In [115]:
finetuned_normalized_forecasts_block_0206 = finetuned_final_forecasts_block_0206[0]

In [116]:
print(finetuned_normalized_forecasts_block_0206)

[0.0, 74.34777636035082, 77.47246021418961, 62.130682641053625, 0.0, 59.26262706700188, 30.450046509615078]


In [117]:
mae_finetuned_block_0206 = finetuned_final_forecasts_block_0206[1]
mae_finetuned_block_0206

[30.809084684601576,
 32.0943890216245,
 PearsonRResult(statistic=0.26026202168412754, pvalue=0.5729762987441772),
 -12.021785481655625]

In [118]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0206[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,0.000000
1,Block0206_2020_08_27,42,41.998810,74.347776
2,Block0206_2020_08_28,39,39.000067,77.472460
3,Block0206_2020_08_31,32,32.000003,62.130683
4,Block0206_2020_09_02,25,25.000352,0.000000
5,Block0206_2020_09_07,23,23.000040,59.262627
6,Block0206_2020_09_16,18,18.000000,30.450047


Block 0302

Predictions wth the frozen model

In [119]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

384/384 [==============================] - 202s 527ms/step


In [120]:
frozen_preds_block_0302.shape

(12288, 7)

In [121]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'frozen_block_0302_VGG19')

In [122]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [123]:
print(frozen_normalized_forecasts_block_0302)

[0.0, 67.75170177541642, 86.26105336425847, 0.00011341862045810558, 0.0, 0.6927550501410074, 0.0]


In [124]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

[39.61712666727335,
 40.96926716897935,
 PearsonRResult(statistic=0.5720766291806804, pvalue=0.17959444044213377),
 -63.25434512952911]

In [125]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,0.000000
1,Block0302_2020_08_27,49,49.000005,67.751702
2,Block0302_2020_08_28,54,53.999570,86.261053
3,Block0302_2020_08_31,50,50.042349,0.000113
4,Block0302_2020_09_02,43,43.000009,0.000000
5,Block0302_2020_09_07,48,48.000572,0.692755
6,Block0302_2020_09_16,37,36.999999,0.000000


Predictions with the finetuned model

In [126]:
# first get the predictions
finetuned_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', finetuned_model)

384/384 [==============================] - 202s 527ms/step


In [127]:
finetuned_preds_block_0302.shape

(12288, 7)

In [128]:
finetuned_final_forecasts_block_0302 = get_final_forecasted_and_true_values(finetuned_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'finetuned_block_0302_VGG19')

In [129]:
finetuned_normalized_forecasts_block_0302 = finetuned_final_forecasts_block_0302[0]

In [130]:
print(finetuned_normalized_forecasts_block_0302)

[0.0, 64.54949078633793, 67.63761545053971, 55.592422457765984, 0.0, 53.03116737416491, 28.21465855369108]


In [131]:
mae_finetuned_block_0302 = finetuned_final_forecasts_block_0302[1]
mae_finetuned_block_0302

[20.085148216445354,
 26.21760934212082,
 PearsonRResult(statistic=0.5195581572394897, pvalue=0.23202943687910713),
 -25.313116360302338]

In [132]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0302[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,0.000000
1,Block0302_2020_08_27,49,49.000005,64.549491
2,Block0302_2020_08_28,54,53.999570,67.637615
3,Block0302_2020_08_31,50,50.042349,55.592422
4,Block0302_2020_09_02,43,43.000009,0.000000
5,Block0302_2020_09_07,48,48.000572,53.031167
6,Block0302_2020_09_16,37,36.999999,28.214659


Block 0303

Predictions wth the frozen model

In [133]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

384/384 [==============================] - 202s 528ms/step


In [134]:
frozen_preds_block_0303.shape

(12288, 7)

In [135]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'frozen_block_0303_VGG19')

In [136]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [137]:
print(frozen_normalized_forecasts_block_0303)

[0.0, 69.18662567510385, 88.44767996951757, 0.0, 0.0, 0.8528166681238647, 0.0]


In [138]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

[35.82592699664251,
 37.00470186529117,
 PearsonRResult(statistic=0.43128111078812376, pvalue=0.3339827250988855),
 -20.841813166280957]

In [139]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,0.000000
1,Block0303_2020_08_27,46,46.000007,69.186626
2,Block0303_2020_08_28,43,42.999982,88.447680
3,Block0303_2020_08_31,39,38.999993,0.000000
4,Block0303_2020_09_02,36,36.025884,0.000000
5,Block0303_2020_09_07,36,36.000000,0.852817
6,Block0303_2020_09_16,23,23.000000,0.000000


Predictions with the finetuned model

In [140]:
# first get the predictions
finetuned_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', finetuned_model)

384/384 [==============================] - 203s 529ms/step


In [141]:
finetuned_preds_block_0303.shape

(12288, 7)

In [142]:
finetuned_final_forecasts_block_0303 = get_final_forecasted_and_true_values(finetuned_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'finetuned_block_0303_VGG19')

In [143]:
finetuned_normalized_forecasts_block_0303 = finetuned_final_forecasts_block_0303[0]

In [144]:
print(finetuned_normalized_forecasts_block_0303)

[0.0, 63.040270879000275, 68.19289415108489, 55.472902971355055, 0.0, 51.94742340428538, 27.650364443147758]


In [145]:
mae_finetuned_block_0303 = finetuned_final_forecasts_block_0303[1]
mae_finetuned_block_0303

[23.47197940698191,
 27.17497662590871,
 PearsonRResult(statistic=0.10351298018805519, pvalue=0.8252108549673645),
 -10.779130330831883]

In [146]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0303[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,0.000000
1,Block0303_2020_08_27,46,46.000007,63.040271
2,Block0303_2020_08_28,43,42.999982,68.192894
3,Block0303_2020_08_31,39,38.999993,55.472903
4,Block0303_2020_09_02,36,36.025884,0.000000
5,Block0303_2020_09_07,36,36.000000,51.947423
6,Block0303_2020_09_16,23,23.000000,27.650364


Block 0304

Predictions wth the frozen model

In [147]:
# first get the predictions
frozen_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', frozen_model)

384/384 [==============================] - 203s 529ms/step


In [148]:
frozen_preds_block_0304.shape

(12288, 7)

In [149]:
frozen_final_forecasts_block_0304 = get_final_forecasted_and_true_values(frozen_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'frozen_block_0304_VGG19')

In [150]:
frozen_normalized_forecasts_block_0304 = frozen_final_forecasts_block_0304[0]

In [151]:
print(frozen_normalized_forecasts_block_0304)

[0.0, 63.96059098664243, 81.62084228442103, 0.0, 0.0, 0.8614548842060378, 0.0]


In [152]:
mae_frozen_block_0304 = frozen_final_forecasts_block_0304[1]
mae_frozen_block_0304

[33.674282626693916,
 34.371752043139644,
 PearsonRResult(statistic=0.5295357871953094, pvalue=0.22157635713674453),
 -31.30438035002156]

In [153]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0304[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,0.000000
1,Block0304_2020_08_27,41,41.002057,63.960591
2,Block0304_2020_08_28,43,42.999998,81.620842
3,Block0304_2020_08_31,42,41.999955,0.000000
4,Block0304_2020_09_02,38,38.000000,0.000000
5,Block0304_2020_09_07,34,34.000000,0.861455
6,Block0304_2020_09_16,24,24.000000,0.000000


Predictions with the finetuned model

In [154]:
# first get the predictions
finetuned_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', finetuned_model)

384/384 [==============================] - 203s 529ms/step


In [155]:
finetuned_preds_block_0304.shape

(12288, 7)

In [156]:
finetuned_final_forecasts_block_0304 = get_final_forecasted_and_true_values(finetuned_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'finetuned_block_0304_VGG19')

In [157]:
finetuned_normalized_forecasts_block_0304 = finetuned_final_forecasts_block_0304[0]

In [158]:
print(finetuned_normalized_forecasts_block_0304)

[0.0, 59.29297295292256, 62.08792070673486, 52.18786659527257, 0.0, 49.3670558032912, 27.739507457378558]


In [159]:
mae_finetuned_block_0304 = finetuned_final_forecasts_block_0304[1]
mae_finetuned_block_0304

[20.239331930799967,
 23.500469667681354,
 PearsonRResult(statistic=0.34400277741485563, pvalue=0.4499349336647049),
 -14.101189539887812]

In [160]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0304[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,0.000000
1,Block0304_2020_08_27,41,41.002057,59.292973
2,Block0304_2020_08_28,43,42.999998,62.087921
3,Block0304_2020_08_31,42,41.999955,52.187867
4,Block0304_2020_09_02,38,38.000000,0.000000
5,Block0304_2020_09_07,34,34.000000,49.367056
6,Block0304_2020_09_16,24,24.000000,27.739507


Block 0305

Predictions wth the frozen model

In [161]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

384/384 [==============================] - 203s 529ms/step


In [162]:
frozen_preds_block_0305.shape

(12288, 7)

In [163]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'frozen_block_0305_VGG19')

In [164]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [165]:
print(frozen_normalized_forecasts_block_0305)

[0.0, 77.95575397605879, 101.41507554497832, 0.0, 0.0, 0.5232143270407809, 0.0]


In [166]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

[38.121087884856614,
 40.437652325843565,
 PearsonRResult(statistic=0.2504761256709438, pvalue=0.5879908729599251),
 -28.588250574469814]

In [167]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,0.000000
1,Block0305_2020_08_27,35,35.000000,77.955754
2,Block0305_2020_08_28,37,37.000000,101.415076
3,Block0305_2020_08_31,30,30.000652,0.000000
4,Block0305_2020_09_02,35,35.001190,0.000000
5,Block0305_2020_09_07,29,28.999994,0.523214
6,Block0305_2020_09_16,20,20.000018,0.000000


Predictions with the finetuned model

In [168]:
# first get the predictions
finetuned_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', finetuned_model)

384/384 [==============================] - 203s 529ms/step


In [169]:
finetuned_preds_block_0305.shape

(12288, 7)

In [170]:
finetuned_final_forecasts_block_0305 = get_final_forecasted_and_true_values(finetuned_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'finetuned_block_0305_VGG19')

In [171]:
finetuned_normalized_forecasts_block_0305 = finetuned_final_forecasts_block_0305[0]

In [172]:
print(finetuned_normalized_forecasts_block_0305)

[0.0, 89.4087658292269, 92.60090820876945, 69.36759112777436, 0.0, 70.37161943135145, 32.068226880174734]


In [173]:
mae_finetuned_block_0305 = finetuned_final_forecasts_block_0305[1]
mae_finetuned_block_0305

[40.54530163961385,
 42.76152312252423,
 PearsonRResult(statistic=-0.21297383562649796, pvalue=0.6465879874034005),
 -32.08672272088277]

In [174]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0305[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,0.000000
1,Block0305_2020_08_27,35,35.000000,89.408766
2,Block0305_2020_08_28,37,37.000000,92.600908
3,Block0305_2020_08_31,30,30.000652,69.367591
4,Block0305_2020_09_02,35,35.001190,0.000000
5,Block0305_2020_09_07,29,28.999994,70.371619
6,Block0305_2020_09_16,20,20.000018,32.068227


Block 0306

Predictions wth the frozen model

In [175]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

384/384 [==============================] - 203s 530ms/step


In [176]:
frozen_preds_block_0306.shape

(12288, 7)

In [177]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'frozen_block_0306_VGG19')

In [178]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [179]:
print(frozen_normalized_forecasts_block_0306)

[0.0, 63.66752102509146, 81.60191096006994, 1.8819018805515952e-05, 0.0, 1.0313099511380037, 0.0]


In [180]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

[33.1768718878578,
 34.29060612119216,
 PearsonRResult(statistic=0.42702810342325703, pvalue=0.33930308031499584),
 -16.826868112555175]

In [181]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,0.000000
1,Block0306_2020_08_27,41,41.000003,63.667521
2,Block0306_2020_08_28,43,43.006851,81.601911
3,Block0306_2020_08_31,40,39.997604,0.000019
4,Block0306_2020_09_02,40,40.000000,0.000000
5,Block0306_2020_09_07,33,32.999982,1.031310
6,Block0306_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [182]:
# first get the predictions
finetuned_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', finetuned_model)

384/384 [==============================] - 203s 529ms/step


In [183]:
finetuned_preds_block_0306.shape

(12288, 7)

In [184]:
finetuned_final_forecasts_block_0306 = get_final_forecasted_and_true_values(finetuned_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'finetuned_block_0306_VGG19')

In [185]:
finetuned_normalized_forecasts_block_0306 = finetuned_final_forecasts_block_0306[0]

In [186]:
print(finetuned_normalized_forecasts_block_0306)

[0.0, 66.68966123426799, 65.45829247587031, 55.34279677564921, 0.0, 53.95559062118672, 30.908587994751088]


In [187]:
mae_finetuned_block_0306 = finetuned_final_forecasts_block_0306[1]
mae_finetuned_block_0306

[25.47927558596076,
 27.481171295642348,
 PearsonRResult(statistic=0.09100963110919476, pvalue=0.8461363099257612),
 -10.44972896449301]

In [188]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0306[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,0.000000
1,Block0306_2020_08_27,41,41.000003,66.689661
2,Block0306_2020_08_28,43,43.006851,65.458292
3,Block0306_2020_08_31,40,39.997604,55.342797
4,Block0306_2020_09_02,40,40.000000,0.000000
5,Block0306_2020_09_07,33,32.999982,53.955591
6,Block0306_2020_09_16,18,18.000000,30.908588
